In [1]:
import glob
import os
import pickle
import yaml
from itertools import cycle
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from neuralhydrology.nh_run import eval_run, finetune
from neuralhydrology.training.train import start_training
from neuralhydrology.utils.config import Config

In [2]:
# Set the device
if torch.cuda.is_available():
    device = "cuda:0"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("Using CPU")

Using GPU: NVIDIA GeForce RTX 4090


In [3]:
# Load the configuration
config_path = Path("config_icond2_gefs.yml")
cfg = Config(config_path)
cfg.device = device

print(f"Loaded config for experiment: {cfg.experiment_name}")
print(f"Run directory: {cfg.run_dir}")

Loaded config for experiment: sequential_forecast_lstm_gefs_icond2_10d_cmal
Run directory: None


## Handle Zarr Cache

We check if a Zarr cache exists from a previous run to avoid rebuilding the dataset. The unified `ForecastDataset` checks for its own caches in `zarr_cache_unified/` first, then falls back to legacy combined caches in `zarr_cache/`.

In [4]:
# Check for per-basin Zarr caches
# ForecastDataset checks for unified caches first, then falls back to legacy caches
zarr_cache_unified = cfg.data_dir / "zarr_cache_unified"
zarr_cache_legacy = cfg.data_dir / "zarr_cache"

if zarr_cache_unified.exists() and any(zarr_cache_unified.glob("*.zarr")):
    print(f"Found unified ForecastDataset Zarr caches in {zarr_cache_unified}")
elif zarr_cache_legacy.exists() and any(zarr_cache_legacy.glob("*_combined.zarr")):
    print(f"Found legacy combined Zarr caches in {zarr_cache_legacy} (will be used as fallback)")
else:
    print(f"No Zarr caches found. Dataset will be rebuilt and cached in {zarr_cache_unified}")

Found legacy combined Zarr caches in ../../data/harz/zarr_cache (will be used as fallback)


In [ ]:
# Start training
# TODO implement Early Stopping
start_training(cfg)

2026-02-13 16:41:01,596: Logging to /home/sngrj0hn/GitHub/neuralhydrology/neuralhydrology/operational_harz/icond2_gefs_10d_sample/runs/sequential_forecast_lstm_gefs_icond2_10d_cmal_1302_164101/output.log initialized.
2026-02-13 16:41:01,596: ### Folder structure created at /home/sngrj0hn/GitHub/neuralhydrology/neuralhydrology/operational_harz/icond2_gefs_10d_sample/runs/sequential_forecast_lstm_gefs_icond2_10d_cmal_1302_164101
2026-02-13 16:41:01,596: ### Run configurations for sequential_forecast_lstm_gefs_icond2_10d_cmal
2026-02-13 16:41:01,597: experiment_name: sequential_forecast_lstm_gefs_icond2_10d_cmal
2026-02-13 16:41:01,597: run_dir: /home/sngrj0hn/GitHub/neuralhydrology/neuralhydrology/operational_harz/icond2_gefs_10d_sample/runs/sequential_forecast_lstm_gefs_icond2_10d_cmal_1302_164101
2026-02-13 16:41:01,597: train_basin_file: basins.txt
2026-02-13 16:41:01,597: validation_basin_file: basins.txt
2026-02-13 16:41:01,598: test_basin_file: basins.txt
2026-02-13 16:41:01,598: t

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x760910fe72f0>>
Traceback (most recent call last):
  File "/home/sngrj0hn/anaconda3/envs/neuralhydrology/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


# Validation: 100%|██████████| 4/4 [00:06<00:00,  1.64s/it]
2026-02-13 16:51:04,941: Epoch 9 average validation loss: 110.80667 -- Median validation metrics: avg_loss: 110.80667, NSE: 0.02653, KGE: -0.01381, Alpha-NSE: 0.25046, Beta-NSE: -0.35803
# Epoch 10: 100%|██████████| 19/19 [00:01<00:00, 15.63it/s, Loss: -184.2835]
2026-02-13 16:51:06,160: Epoch 10 average loss: avg_loss: -216.51183, avg_total_loss: -216.51183
Metrics for 1h are calculated over last 1 elements only. Ignoring 239 predictions per sequence.
# Validation: 100%|██████████| 4/4 [00:06<00:00,  1.55s/it]
2026-02-13 16:51:12,836: Epoch 10 average validation loss: 176.64260 -- Median validation metrics: avg_loss: 176.64260, NSE: -0.02681, KGE: -0.03491, Alpha-NSE: 0.24036, Beta-NSE: -0.46453
# Epoch 11: 100%|██████████| 19/19 [00:01<00:00, 15.71it/s, Loss: -270.0695]
2026-02-13 16:51:14,047: Epoch 11 average loss: avg_loss: -187.56069, avg_total_loss: -187.56069
Metrics for 1h are calculated over last 1 elements only. Ign

## Evaluation and Visualization

The following cells evaluate the trained model on the test and train sets, and then visualize the results using interactive Plotly charts. This includes:
1.  **Training Period Performance**: Checking how well the model learned the training data (Day 1 forecast).
2.  **Interactive Forecast Slider**: Viewing predictions for different forecast horizons (Day 1 to Day 10).
3.  **Spaghetti Plot**: Visualizing all forecast traces over time against observations.

In [ ]:
# Find the latest run directory
experiment_name = cfg.experiment_name
run_dirs = glob.glob(f"runs/{experiment_name}*")

if not run_dirs:
    raise FileNotFoundError(f"No run directories found for experiment: {experiment_name}")

# Sort by modification time to ensure we get the actual latest run
run_dirs.sort(key=os.path.getmtime)
latest_run_dir = Path(run_dirs[-1])
print(f"Evaluating run at: {latest_run_dir}")

In [ ]:
# Run evaluation on test and train sets
print("Starting Test Evaluation...")
eval_run(run_dir=latest_run_dir, period="test")

print("Starting Train Evaluation...")
eval_run(run_dir=latest_run_dir, period="train")

In [ ]:
# Load Test Results
test_results_files = list(latest_run_dir.glob("test/*/test_results.p"))
if not test_results_files:
    raise FileNotFoundError("No test results found.")

test_results_file = test_results_files[0]
print(f"Loading test results from: {test_results_file}")

with open(test_results_file, "rb") as fp:
    results = pickle.load(fp)
    
print(f"Available basins: {list(results.keys())}")

# Set the basin variable for the plots below
basin = 'DE4' 
if basin not in results:
    basin = list(results.keys())[0]
    print(f"Basin 'DE4' not found, defaulting to {basin}")
else:
    print(f"Basin set to: {basin}")

In [ ]:
# --- Training Period: Day 1 Forecast Plot ---

# 1. Load Training Results
train_results_files = list(latest_run_dir.glob("train/*/train_results.p"))
if not train_results_files:
    print("No training results found. Make sure eval_run(..., period='train') has completed.")
else:
    train_results_file = train_results_files[0]
    print(f"Loading training results from: {train_results_file}")
    
    with open(train_results_file, "rb") as fp:
        train_results = pickle.load(fp)

    if 'basin' in locals():
        # Assuming 1h frequency as per config
        freq = list(train_results[basin].keys())[0]
        ds_train = train_results[basin][freq]['xr']
        
        # 2. Subsample to avoid overlaps (Daily Stride)
        dates_train = pd.to_datetime(ds_train['date'].values)
        time_diff_train = dates_train[1] - dates_train[0]
        
        if time_diff_train < pd.Timedelta('24h'):
            stride_train = int(pd.Timedelta('24h') / time_diff_train)
            ds_train_subset = ds_train.isel(date=slice(0, None, stride_train))
        else:
            ds_train_subset = ds_train

        # 3. Select Day 1 (First 24 hours)
        ds_train_day1 = ds_train_subset.isel(time_step=slice(0, 24))

        # 4. Stack
        ds_train_flat = ds_train_day1.stack(combined=('date', 'time_step'))

        # 5. Extract
        if 'samples' in ds_train_flat['discharge_vol_sim'].dims:
            qsim_train = ds_train_flat['discharge_vol_sim'].transpose('combined', 'samples').values
        else:
            qsim_train = ds_train_flat['discharge_vol_sim'].values
        
        qobs_train = ds_train_flat['discharge_vol_obs'].values

        # 6. Reconstruct Dates
        dates_train_flat = pd.to_datetime(ds_train_flat['date'].values) + pd.to_timedelta(ds_train_flat['time_step'].values, unit='h')
        
        # 7. Sort
        sort_idx_train = np.argsort(dates_train_flat)
        dates_train_flat = dates_train_flat[sort_idx_train]
        qsim_train = qsim_train[sort_idx_train]
        qobs_train = qobs_train[sort_idx_train]

        # 8. Handle NaNs
        nan_mask_train = np.all(np.isnan(qsim_train), axis=-1) if qsim_train.ndim == 2 else np.isnan(qsim_train)
        if np.any(nan_mask_train):
            valid_mask_train = ~nan_mask_train
            dates_train_flat = dates_train_flat[valid_mask_train]
            qsim_train = qsim_train[valid_mask_train]
            qobs_train = qobs_train[valid_mask_train]

        # 9. Calculate Percentiles
        if qsim_train.ndim == 2:
            y_median_train = np.nanmedian(qsim_train, axis=-1)
            y_05_train = np.nanpercentile(qsim_train, 5, axis=-1)
            y_95_train = np.nanpercentile(qsim_train, 95, axis=-1)
            y_25_train = np.nanpercentile(qsim_train, 25, axis=-1)
            y_75_train = np.nanpercentile(qsim_train, 75, axis=-1)
        else:
            y_median_train = qsim_train
            y_05_train = y_95_train = y_25_train = y_75_train = qsim_train

        # 10. Plot
        fig = go.Figure()

        # 90% CI
        fig.add_trace(go.Scatter(
            x=np.concatenate([dates_train_flat, dates_train_flat[::-1]]),
            y=np.concatenate([y_95_train, y_05_train[::-1]]),
            fill='toself',
            fillcolor='rgba(53, 183, 121, 0.5)',
            line=dict(color='rgba(255,255,255,0)'),
            name='90% CI (5-95)',
            showlegend=True
        ))

        # 50% CI
        fig.add_trace(go.Scatter(
            x=np.concatenate([dates_train_flat, dates_train_flat[::-1]]),
            y=np.concatenate([y_75_train, y_25_train[::-1]]),
            fill='toself',
            fillcolor='rgba(68, 1, 84, 0.5)',
            line=dict(color='rgba(255,255,255,0)'),
            name='50% CI (25-75)',
            showlegend=True
        ))

        # Median
        fig.add_trace(go.Scatter(
            x=dates_train_flat,
            y=y_median_train,
            mode='lines',
            line=dict(color='red', width=2),
            name='Median'
        ))

        # Observed
        fig.add_trace(go.Scatter(
            x=dates_train_flat,
            y=qobs_train,
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            name='Observed'
        ))

        fig.update_layout(
            title='Training Period: Discharge Prediction (Day 1 Ahead)',
            xaxis_title='Date',
            yaxis_title='Discharge [m³/s]',
            template='plotly_white',
            hovermode='x unified'
        )

        fig.show()
    else:
        print("Basin variable not defined.")

In [ ]:
# --- Data Preparation with Slider Support ---

if 'results' in locals() and 'basin' in locals():
    # Assuming 1h frequency as per config
    freq = list(results[basin].keys())[0]
    ds = results[basin][freq]['xr']
    
    # 1. Subsample to avoid overlaps (Daily Stride)
    dates = pd.to_datetime(ds['date'].values)
    time_diff = dates[1] - dates[0]
    if time_diff < pd.Timedelta('24h'):
        stride = int(pd.Timedelta('24h') / time_diff)
        ds_subset = ds.isel(date=slice(0, None, stride))
    else:
        ds_subset = ds

    # Determine number of days in forecast horizon
    n_steps = len(ds_subset['time_step'])
    n_days = n_steps // 24
    print(f"Generating interactive plot for {n_days} forecast days...")
    
    # Number of traces per day: 90% CI, 50% CI, Median, Observed, Persistence = 5
    traces_per_day = 5

    fig = go.Figure()
    steps = []

    # Loop through each day (0 to n_days-1) to generate traces
    for day_idx in range(n_days):
        # Select Day slice (e.g., 0-24, 24-48, etc.)
        start_step = day_idx * 24
        end_step = (day_idx + 1) * 24
        
        # Slice, Stack, Extract
        ds_day = ds_subset.isel(time_step=slice(start_step, end_step))
        ds_flat = ds_day.stack(combined=('date', 'time_step'))

        if 'samples' in ds_flat['discharge_vol_sim'].dims:
            qsim = ds_flat['discharge_vol_sim'].transpose('combined', 'samples').values
        else:
            qsim = ds_flat['discharge_vol_sim'].values
        
        qobs = ds_flat['discharge_vol_obs'].values

        # Reconstruct Dates
        dates_flat = pd.to_datetime(ds_flat['date'].values) + pd.to_timedelta(ds_flat['time_step'].values, unit='h')
        
        # Sort
        sort_idx = np.argsort(dates_flat)
        dates_flat = dates_flat[sort_idx]
        qsim = qsim[sort_idx]
        qobs = qobs[sort_idx]

        # Handle NaNs
        nan_mask = np.all(np.isnan(qsim), axis=-1) if qsim.ndim == 2 else np.isnan(qsim)
        if np.any(nan_mask):
            valid_mask = ~nan_mask
            dates_flat = dates_flat[valid_mask]
            qsim = qsim[valid_mask]
            qobs = qobs[valid_mask]

        # Calculate Percentiles
        if qsim.ndim == 2:
            y_median = np.nanmedian(qsim, axis=-1)
            y_05 = np.nanpercentile(qsim, 5, axis=-1)
            y_95 = np.nanpercentile(qsim, 95, axis=-1)
            y_25 = np.nanpercentile(qsim, 25, axis=-1)
            y_75 = np.nanpercentile(qsim, 75, axis=-1)
        else:
            y_median = qsim
            y_05 = y_95 = y_25 = y_75 = qsim

        # Calculate persistence (24-hour shifted observations)
        # Persistence = what was observed 24 hours earlier
        persistence = np.full_like(qobs, np.nan)
        if day_idx == 0:
            # Day 0: need previous day's observations (hours 0-23 from yesterday)
            # For test period visualization, we shift the entire time series by 24 hours
            persistence[24:] = qobs[:-24]
        else:
            # Day 1+: persistence is already captured by shifting by 24h
            persistence[24:] = qobs[:-24]

        # Visibility: Only Day 1 (index 0) is visible initially
        is_visible = (day_idx == 0)

        # Add Traces (5 traces per day)
        
        # 1. 90% CI
        fig.add_trace(go.Scatter(
            x=np.concatenate([dates_flat, dates_flat[::-1]]),
            y=np.concatenate([y_95, y_05[::-1]]),
            fill='toself',
            fillcolor='rgba(53, 183, 121, 0.5)',
            line=dict(color='rgba(255,255,255,0)'),
            name='90% CI (5-95)',
            visible=is_visible,
            showlegend=True
        ))

        # 2. 50% CI
        fig.add_trace(go.Scatter(
            x=np.concatenate([dates_flat, dates_flat[::-1]]),
            y=np.concatenate([y_75, y_25[::-1]]),
            fill='toself',
            fillcolor='rgba(68, 1, 84, 0.5)',
            line=dict(color='rgba(255,255,255,0)'),
            name='50% CI (25-75)',
            visible=is_visible,
            showlegend=True
        ))

        # 3. Median
        fig.add_trace(go.Scatter(
            x=dates_flat,
            y=y_median,
            mode='lines',
            line=dict(color='red', width=2),
            name='Median',
            visible=is_visible,
            showlegend=True
        ))

        # 4. Observed
        fig.add_trace(go.Scatter(
            x=dates_flat,
            y=qobs,
            mode='lines',
            line=dict(color='black', width=2, dash='dash'),
            name='Observed',
            visible=is_visible,
            showlegend=True
        ))
        
        # 5. Persistence
        fig.add_trace(go.Scatter(
            x=dates_flat,
            y=persistence,
            mode='lines',
            line=dict(color='orange', width=2, dash='dot'),
            name='Persistence',
            visible=is_visible,
            showlegend=True
        ))

    # Create Slider Steps
    # Total traces = n_days * traces_per_day
    for i in range(n_days):
        step = dict(
            method="update",
            args=[{"visible": [False] * (n_days * traces_per_day)},
                  {"title": f"Discharge Prediction - Day {i + 1} Ahead"}],
            label=str(i + 1)
        )
        # Enable the traces for this day
        for j in range(traces_per_day):
            step["args"][0]["visible"][i * traces_per_day + j] = True
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "Forecast Day: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        sliders=sliders,
        title='Discharge Prediction - Day 1 Ahead',
        xaxis_title='Date',
        yaxis_title='Discharge [m³/s]',
        template='plotly_white',
        hovermode='x unified'
    )

    fig.show()
else:
    print("Please run the previous cells to load 'results' and define 'basin'.")

In [ ]:
# --- Discharge Prediction GIF (Matplotlib) ---
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from PIL import Image
import io

# Set Times New Roman font
plt.rcParams["font.family"] = "Times New Roman"

if 'results' in locals() and 'basin' in locals():
    # Reuse ds_subset from slider cell or re-derive
    if 'ds_subset' not in locals():
        freq = list(results[basin].keys())[0]
        ds = results[basin][freq]['xr']
        dates = pd.to_datetime(ds['date'].values)
        time_diff = dates[1] - dates[0]
        if time_diff < pd.Timedelta('24h'):
            stride = int(pd.Timedelta('24h') / time_diff)
            ds_subset = ds.isel(date=slice(0, None, stride))
        else:
            ds_subset = ds

    # Determine number of days
    n_steps = len(ds_subset['time_step'])
    n_days = n_steps // 24
    print(f"Creating GIF for {n_days} forecast days...")

    # Pre-extract data for all days
    frames_data = []
    
    for day_idx in range(n_days):
        start_step = day_idx * 24
        end_step = (day_idx + 1) * 24
        
        ds_day = ds_subset.isel(time_step=slice(start_step, end_step))
        ds_flat = ds_day.stack(combined=('date', 'time_step'))

        if 'samples' in ds_flat['discharge_vol_sim'].dims:
            qsim = ds_flat['discharge_vol_sim'].transpose('combined', 'samples').values
        else:
            qsim = ds_flat['discharge_vol_sim'].values
        
        qobs = ds_flat['discharge_vol_obs'].values

        # Reconstruct dates
        dates_flat = pd.to_datetime(ds_flat['date'].values) + pd.to_timedelta(ds_flat['time_step'].values, unit='h')
        
        # Sort
        sort_idx = np.argsort(dates_flat)
        dates_flat = dates_flat[sort_idx]
        qsim = qsim[sort_idx]
        qobs = qobs[sort_idx]

        # Handle NaNs
        nan_mask = np.all(np.isnan(qsim), axis=-1) if qsim.ndim == 2 else np.isnan(qsim)
        if np.any(nan_mask):
            valid_mask = ~nan_mask
            dates_flat = dates_flat[valid_mask]
            qsim = qsim[valid_mask]
            qobs = qobs[valid_mask]

        # Calculate percentiles
        if qsim.ndim == 2:
            y_median = np.nanmedian(qsim, axis=-1)
            y_05 = np.nanpercentile(qsim, 5, axis=-1)
            y_95 = np.nanpercentile(qsim, 95, axis=-1)
            y_25 = np.nanpercentile(qsim, 25, axis=-1)
            y_75 = np.nanpercentile(qsim, 75, axis=-1)
        else:
            y_median = qsim
            y_05 = y_95 = y_25 = y_75 = qsim

        # Persistence (24-hour shifted observations)
        persistence = np.full_like(qobs, np.nan)
        persistence[24:] = qobs[:-24]

        frames_data.append({
            'dates': dates_flat,
            'y_median': y_median,
            'y_05': y_05,
            'y_95': y_95,
            'y_25': y_25,
            'y_75': y_75,
            'qobs': qobs,
            'persistence': persistence
        })

    # Get global y-axis limits for consistent scaling
    all_y_values = []
    for data in frames_data:
        all_y_values.extend(data['y_95'][~np.isnan(data['y_95'])])
        all_y_values.extend(data['y_05'][~np.isnan(data['y_05'])])
        all_y_values.extend(data['qobs'][~np.isnan(data['qobs'])])
    y_min, y_max = np.nanmin(all_y_values), np.nanmax(all_y_values)
    y_padding = (y_max - y_min) * 0.1

    # Generate frames as PIL Images
    pil_frames = []
    
    for day_idx in range(n_days):
        fig, ax = plt.subplots(figsize=(8, 5))
        fig.patch.set_facecolor('white')
        ax.patch.set_facecolor('white')
        
        data = frames_data[day_idx]
        
        # Plot 90% CI (fill_between) - Green
        ax.fill_between(data['dates'], data['y_05'], data['y_95'],
                        alpha=0.5, color='C2', label='90% CI')
        
        # Plot 50% CI - Purple
        ax.fill_between(data['dates'], data['y_25'], data['y_75'],
                        alpha=0.5, color='C4', label='50% CI')
        
        # Median - Red
        ax.plot(data['dates'], data['y_median'], 'C3-', lw=2, label='Median')
        
        # Observed - Black dashed
        ax.plot(data['dates'], data['qobs'], 'k--', lw=2, label='Observed')
        
        # Persistence - Orange dotted
        ax.plot(data['dates'], data['persistence'], 'C1:', lw=2, label='Persistence')
        
        ax.set_title(f'Discharge Prediction - Day {day_idx + 1}', fontweight='bold', fontsize=14)
        ax.set_xlabel('Date', fontsize=12)
        ax.set_ylabel('Discharge [m³/s]', fontsize=12)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(y_min - y_padding, y_max + y_padding)
        
        # Format x-axis dates
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        
        fig.tight_layout()
        
        # Save to buffer and convert to PIL Image
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=150, facecolor='white', edgecolor='none')
        buf.seek(0)
        pil_frames.append(Image.open(buf).copy())
        buf.close()
        plt.close(fig)
        
        print(f"  Frame {day_idx + 1}/{n_days} created")

    # Save GIF with proper disposal (each frame replaces the previous)
    gif_path = 'discharge_prediction.gif'
    pil_frames[0].save(
        gif_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=1000,  # 1 second per frame
        loop=0,  # Loop forever
        disposal=2  # Clear frame before drawing next (prevents ghosting)
    )
    
    print(f"\nGIF saved to: {gif_path}")
    
    # Display the first frame as preview
    fig_preview, ax_preview = plt.subplots(figsize=(8, 5))
    data = frames_data[0]
    ax_preview.fill_between(data['dates'], data['y_05'], data['y_95'], alpha=0.5, color='C2', label='90% CI')
    ax_preview.fill_between(data['dates'], data['y_25'], data['y_75'], alpha=0.5, color='C4', label='50% CI')
    ax_preview.plot(data['dates'], data['y_median'], 'C3-', lw=2, label='Median')
    ax_preview.plot(data['dates'], data['qobs'], 'k--', lw=2, label='Observed')
    ax_preview.plot(data['dates'], data['persistence'], 'C1:', lw=2, label='Persistence')
    ax_preview.set_title('Discharge Prediction - Day 1 (Preview)', fontweight='bold', fontsize=14)
    ax_preview.set_xlabel('Date', fontsize=12)
    ax_preview.set_ylabel('Discharge [m³/s]', fontsize=12)
    ax_preview.legend(loc='upper right', fontsize=9)
    ax_preview.grid(True, alpha=0.3)
    ax_preview.set_ylim(y_min - y_padding, y_max + y_padding)
    ax_preview.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax_preview.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax_preview.xaxis.get_majorticklabels(), rotation=45, ha='right')
    fig_preview.tight_layout()
    plt.show()
else:
    print("Please run the previous cells to load 'results' and define 'basin'.")


In [ ]:
# --- Spaghetti Plot: All Forecast Traces (Median) ---

if 'results' in locals() and 'basin' in locals():
    # Use ds_subset from previous cell if available, otherwise re-derive
    if 'ds_subset' not in locals():
        # Assuming 1h frequency as per config
        freq = list(results[basin].keys())[0]
        ds = results[basin][freq]['xr']
        dates = pd.to_datetime(ds['date'].values)
        time_diff = dates[1] - dates[0]
        if time_diff < pd.Timedelta('24h'):
            stride = int(pd.Timedelta('24h') / time_diff)
            ds_subset = ds.isel(date=slice(0, None, stride))
        else:
            ds_subset = ds

    fig = go.Figure()

    # 1. Plot Individual Forecast Traces
    n_forecasts = len(ds_subset['date'])
    print(f"Plotting {n_forecasts} forecast traces...")
    
    # Use Plotly standard qualitative colors, cycling through them
    palette = cycle(px.colors.qualitative.Plotly)
    colors = [next(palette) for _ in range(n_forecasts)]

    # Pre-calculate time steps in hours
    time_steps_hours = pd.to_timedelta(ds_subset['time_step'].values, unit='h')
    n_time_steps = len(time_steps_hours)
    
    # --- Prepare data for CSV export ---
    forecast_medians = []  # List to store median forecasts
    issue_dates = []       # List to store issue dates
    
    for i, date in enumerate(ds_subset['date'].values):
        # Extract forecast for this date
        # Dimensions: (time_step, samples) -> we want median over samples
        forecast_slice = ds_subset['discharge_vol_sim'].isel(date=i)
        
        if 'samples' in forecast_slice.dims:
            # Calculate median across samples
            forecast_median = forecast_slice.median(dim='samples').values
        else:
            forecast_median = forecast_slice.values
            
        # Construct x-axis (Date + Time Steps)
        start_date = pd.to_datetime(date)
        forecast_dates = start_date + time_steps_hours
        
        # Store for CSV export
        issue_dates.append(start_date)
        forecast_medians.append(forecast_median)
        
        # Handle NaNs for plotting
        if np.all(np.isnan(forecast_median)):
            continue

        # Add Trace with unique color from Plotly palette
        fig.add_trace(go.Scatter(
            x=forecast_dates,
            y=forecast_median,
            mode='lines',
            line=dict(width=1.5, color=colors[i]), 
            opacity=0.8,
            showlegend=False, 
            name=f'Forecast {start_date.strftime("%Y-%m-%d")}'
        ))

    # 2. Plot Observations (Ground Truth)
    # We reconstruct the observations from the first day (0-24h) of each forecast
    # to ensure we have the full continuous time series starting from the first forecast date.
    
    obs_list = []
    date_list = []
    
    for i in range(len(ds_subset['date'])):
        # Take first 24 steps (Day 1) to stitch together the continuous timeline
        obs_slice = ds_subset['discharge_vol_obs'].isel(date=i, time_step=slice(0, 24)).values
        d_slice = pd.to_datetime(ds_subset['date'].values[i]) + time_steps_hours[:24]
        
        obs_list.append(obs_slice)
        date_list.append(d_slice)
        
    flat_obs = np.concatenate(obs_list)
    flat_dates = np.concatenate(date_list)
    
    # Sort to be sure
    sort_idx = np.argsort(flat_dates)
    flat_dates = flat_dates[sort_idx]
    flat_obs = flat_obs[sort_idx]
    
    fig.add_trace(go.Scatter(
        x=flat_dates,
        y=flat_obs,
        mode='lines',
        line=dict(color='black', width=2),
        name='Observed'
    ))

    fig.update_layout(
        title='All Forecast Traces (Median) vs Observed',
        xaxis_title='Date',
        yaxis_title='Discharge [m³/s]',
        template='plotly_white',
        hovermode='x unified'
    )

    fig.show()
    
    # --- Export median forecasts to CSV ---
    # Create DataFrame with issue times as rows and lead times (1-240) as columns
    forecast_df = pd.DataFrame(
        forecast_medians,
        index=pd.to_datetime(issue_dates),
        columns=[str(h) for h in range(1, n_time_steps + 1)]
    )
    forecast_df.index.name = 'issue_time'
    
    # Export to CSV
    csv_path = f'median_forecasts_{basin}.csv'
    forecast_df.to_csv(csv_path)
    print(f"\nExported median forecasts to: {csv_path}")
    print(f"  Shape: {forecast_df.shape[0]} issue times x {forecast_df.shape[1]} lead times (hours)")
    print(f"  Issue time range: {forecast_df.index.min()} to {forecast_df.index.max()}")
    
else:
    print("Please run the previous cells to load 'results' and define 'basin'.")

## MAE Improvement Over Persistence Benchmark

This section calculates the Mean Absolute Error (MAE) improvement of the model over a persistence benchmark for each basin and forecast day.

**Methodology:**
1. **Daily aggregation**: Hourly values are first averaged to daily mean values for each forecast date
2. **MAE calculation**: MAE is then computed on the daily aggregated values across all forecast dates

**Persistence benchmark**: The observation time series shifted by 24 hours. For any forecast day, the persistence "forecast" is the mean of observations from 24 hours earlier.

- **Day 0** (hours 0-23): Uses observations from the previous forecast date (hours 0-23)
- **Day 1+** (hours 24+): Uses observations from 24 hours earlier within the current forecast

This represents a naive forecast that assumes "the next 24 hours will be like the previous 24 hours."

**Improvement calculation**: `(MAE_persistence - MAE_model) / MAE_persistence * 100`

Note: For Day 0, the first forecast date is excluded since there is no "previous day" available for it.

In [ ]:
# --- MAE Improvement Over Persistence Benchmark ---

# Basin name mapping for display
basin_names = {
    'DE1': 'Innerste Res Inflow',
    'DE2': 'Oker Res Inflow',
    'DE3': 'Ecker Res Inflow',
    'DE4': 'Soese Res Inflow',
    'DE5': 'Grane Res Inflow'
}

# Number of forecast days (10 days = 240 hours)
n_days = 10

# Store results: {basin: [improvement_day0, improvement_day1, ...]}
improvements = {basin_id: [] for basin_id in results.keys()}

for basin_id in results.keys():
    freq = list(results[basin_id].keys())[0]
    ds = results[basin_id][freq]['xr']

    # Subsample to daily stride (one forecast per day)
    dates = pd.to_datetime(ds['date'].values)
    time_diff = dates[1] - dates[0] if len(dates) > 1 else pd.Timedelta('24h')
    if time_diff < pd.Timedelta('24h'):
        stride = int(pd.Timedelta('24h') / time_diff)
        ds = ds.isel(date=slice(0, None, stride))

    # Get observations and simulations
    obs = ds['discharge_vol_obs'].values  # (n_dates, n_timesteps)

    if 'samples' in ds['discharge_vol_sim'].dims:
        sim = ds['discharge_vol_sim'].median(dim='samples').values
    else:
        sim = ds['discharge_vol_sim'].values

    # Calculate per forecast day (0-9)
    # Persistence = 24-hour shifted observation time series
    for day_idx in range(n_days):
        start_hour = day_idx * 24
        end_hour = (day_idx + 1) * 24

        if end_hour > obs.shape[1]:
            improvements[basin_id].append(np.nan)
            continue

        if day_idx == 0:
            # Day 0: persistence uses previous forecast date's hours 0-23
            # Skip first forecast date since we need "yesterday's" observations
            obs_day = obs[1:, start_hour:end_hour]  # (n_dates-1, 24)
            sim_day = sim[1:, start_hour:end_hour]  # (n_dates-1, 24)
            persistence_day = obs[:-1, 0:24]  # Previous date's hours 0-23
        else:
            # Day 1+: persistence = observations from 24 hours earlier (within same forecast)
            obs_day = obs[:, start_hour:end_hour]  # (n_dates, 24)
            sim_day = sim[:, start_hour:end_hour]  # (n_dates, 24)
            pers_start = (day_idx - 1) * 24
            pers_end = day_idx * 24
            persistence_day = obs[:, pers_start:pers_end]  # Hours from 24h earlier

        # Aggregate hourly to daily (mean over 24 hours for each forecast date)
        obs_daily = np.nanmean(obs_day, axis=1)       # (n_dates,)
        sim_daily = np.nanmean(sim_day, axis=1)       # (n_dates,)
        pers_daily = np.nanmean(persistence_day, axis=1)  # (n_dates,)

        # Remove dates with any NaN in original hourly data
        valid_mask = (
            ~np.any(np.isnan(obs_day), axis=1) &
            ~np.any(np.isnan(sim_day), axis=1) &
            ~np.any(np.isnan(persistence_day), axis=1)
        )

        obs_daily = obs_daily[valid_mask]
        sim_daily = sim_daily[valid_mask]
        pers_daily = pers_daily[valid_mask]

        if len(obs_daily) == 0:
            improvements[basin_id].append(np.nan)
            continue

        # Calculate MAE on daily aggregated values
        mae_model = np.mean(np.abs(sim_daily - obs_daily))
        mae_persistence = np.mean(np.abs(pers_daily - obs_daily))

        # Calculate improvement (%)
        if mae_persistence > 0:
            improvement = (mae_persistence - mae_model) / mae_persistence * 100
        else:
            improvement = 0.0

        improvements[basin_id].append(improvement)

# Create grouped bar chart
fig = go.Figure()

# Color palette matching the reference image
colors = {
    'DE3': '#2d6a4f',  # Ecker - dark teal
    'DE2': '#74c69d',  # Oker - light teal
    'DE1': '#a8dadc',  # Innerste - pale blue
    'DE5': '#52b788',  # Grane - green
    'DE4': '#5a189a',  # Soese - purple
}

# Add bars for each basin (order to match reference image, only include available basins)
for basin_id in ['DE3', 'DE2', 'DE1', 'DE5', 'DE4']:
    if basin_id in improvements:
        fig.add_trace(go.Bar(
            name=basin_names[basin_id],
            x=list(range(n_days)),
            y=improvements[basin_id],
            marker_color=colors[basin_id]
        ))

fig.update_layout(
    title='HydroForecast Mean Absolute Error Improvement Over Persistence',
    xaxis_title='Forecast Day',
    yaxis_title='Percent Improvement (%)',
    barmode='group',
    template='plotly_white',
    yaxis=dict(range=[0, 60], tickvals=[0, 20, 40, 60], ticktext=['0%', '20%', '40%', '60%']),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5)
)

fig.show()

# Print summary table
print("\nMAE Improvement Over Persistence (%):")
print("-" * 100)
header = "Basin".ljust(25) + "".join([f"Day {i}".rjust(8) for i in range(n_days)])
print(header)
print("-" * 100)
for basin_id in results.keys():
    row = basin_names.get(basin_id, basin_id).ljust(25) + "".join([
        f"{v:.1f}%".rjust(8) if not np.isnan(v) else "N/A".rjust(8) 
        for v in improvements[basin_id]
    ])
    print(row)